# Data Cleaning

Prepare the full merged item table, inspect data quality, exclude whole orders using fixed eligibility rules, and verify the retained dataset before saving. The final table contains the 25 original columns, including price and freight, for eligible delivered orders. Quality flags and exclusion details remain in memory for inspection.

Preparation and temporal tables below describe the full input. The filtering flow and retained-dataset tables show how that input becomes the final clean result. The notebook produces dataset files and displayed results; documentation is maintained separately.


## 1. Input and execution context

Locate the repository from its root, `notebooks`, or a nested working directory. Load the schema-aware full merge and capture an in-memory baseline before any transformation.


In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display


def find_project_root(start_path: Path) -> Path:
    """Find the repository containing src/ and requirements.txt."""
    for path in [start_path, *start_path.parents]:
        if (path / "src").is_dir() and (path / "requirements.txt").is_file():
            return path
    raise FileNotFoundError(
        "Could not locate the project root containing src/ and requirements.txt."
    )


PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [ ]:
from src.cleaning import (
    CLEANED_DATASET_NAME,
    assert_cleaned_roundtrip,
    clean_order_items,
    format_cleaning_preview,
    inspect_text_fields,
    save_cleaned_data,
)
from src.cleaning.validation import (
    assert_input_unchanged,
    capture_input_baseline,
    validate_cleaning_input,
)
from src.data_loader import load_data
from src.pipeline.contracts.schema import BENCHMARK_ROW_COUNT, FULL_MERGE_SCHEMA
from src.paths import INTERIM_DIR, PROCESSED_DIR

print(f"Project root: {PROJECT_ROOT}")
print(f"Interim input: {INTERIM_DIR}")
print(f"Processed output: {PROCESSED_DIR}")


### Load the full merged table

Read `merged_order_items` from `interim`. Parquet is preferred with CSV fallback; add `file_format="csv"` to read CSV directly. The reference row count describes the current source dataset, while schema and key checks enforce structural validity.


In [ ]:
df = load_data("merged_order_items", stage="interim")
input_baseline = capture_input_baseline(df)
input_summary = pd.DataFrame([
    ["Item rows", BENCHMARK_ROW_COUNT, len(df)],
    ["Columns", len(FULL_MERGE_SCHEMA.columns), len(df.columns)],
], columns=["metric", "reference", "observed"])
input_summary["Matches reference"] = input_summary["reference"].eq(input_summary["observed"])
display(format_cleaning_preview(input_summary))


## 2. Input integrity

Inspect text formats and source domains, then enforce schema, key and entity consistency. Repeated entity IDs are expected at item grain. Invalid observed text values stop validation; optional missing text attributes are reported separately.


### Text formats and source domains

String IDs must contain 32 lowercase hexadecimal characters; ZIP prefixes must contain five ASCII digits. State and status checks use fixed source domains. All invalid cells remain available in `text_inspection["issues"]`, with original values and positional references.


In [ ]:
text_inspection = inspect_text_fields(df)
with pd.option_context("display.max_columns", None):
    display(format_cleaning_preview(text_inspection["summary"]))
    if not text_inspection["issues"].empty:
        print("First ten invalid cells; full records: text_inspection['issues']")
        display(format_cleaning_preview(text_inspection["issues"]))


### Keys and repeated entity attributes

Check unique item keys, mandatory identifiers and consistency of repeated order, customer, product and seller attributes before cleaning.


In [ ]:
input_validation = validate_cleaning_input(df)
print("Item and entity key profiles")
display(format_cleaning_preview(input_validation["key_profile"]))
print("Entity attribute consistency")
display(format_cleaning_preview(input_validation["entity_consistency"]))


## 3. Preparation of the full input

Run the reusable workflow once. It validates input, normalizes city whitespace, handles observed non-positive physical cells, computes quality flags on the full input and filters whole orders. The returned diagnostics preserve the preparation, exclusion and final-validation results for the sections below.


In [ ]:
cleaned_df, cleaning_details = clean_order_items(df)


### City whitespace changes before filtering

Trim city edges and collapse internal whitespace runs. Counts here describe changes across the full input, including orders subsequently excluded.


In [ ]:
text_change_summary = cleaning_details["text_changes"]
display(format_cleaning_preview(text_change_summary))


### Physical measurements before filtering

Only observed weight or dimensions at or below zero become missing during preparation. Their items receive a physical-quality flag used to exclude the whole order. Pre-existing nulls in optional attributes remain allowed. Price, zero freight and numeric tails retain their original values.


In [ ]:
measurement_summary = cleaning_details["measurements"]
display(format_cleaning_preview(measurement_summary))
print("Preparation and quality-flag validation")
display(format_cleaning_preview(cleaning_details["inspection_checks"]))


## 4. Temporal quality of the full input

Use complete timestamp precision. Missing endpoints are counted as unassessable in inspection; the eligibility step separately excludes orders lacking a required core or estimated-delivery timestamp.


### Core timestamp sequence at order grain

Check Purchase → Approved → Carrier Handoff → Customer Delivery. A confirmed reversal flags the order and all its items. These summaries use distinct orders from the full input.


In [ ]:
timeline_summary = cleaning_details["timeline"]
core_timestamp_summary = cleaning_details["core_timestamps"]
display(format_cleaning_preview(timeline_summary))
display(format_cleaning_preview(core_timestamp_summary))


### Item-level shipping deadlines

Review deadlines before purchase and deadline years beyond the latest purchase year in the full input. The latter is an observation-based review signal. Under the agreed strict policy, either observed signal excludes the whole order. The comparison is computed before filtering and original timestamps remain available in the diagnostics.


In [ ]:
deadline_summary = cleaning_details["deadlines"]
deadline_review_records = cleaning_details["deadline_records"]
display(format_cleaning_preview(deadline_summary))
if not deadline_review_records.empty:
    print("First ten review items; full records: deadline_review_records")
    with pd.option_context("display.max_columns", None):
        display(format_cleaning_preview(deadline_review_records))


## 5. Whole-order eligibility

Exclude a whole order if any item has a non-delivered status, a missing required timestamp, a core timestamp reversal, an invalid physical measurement or a deadline-review flag. Required timestamps are Purchase, Approved, Carrier Handoff, Customer Delivery and Estimated Delivery. Every item of a retained order is kept.


### Affected orders by reason

Each reason is counted independently against the full input. Affected-order and item counts can overlap across reasons, so they should not be added to obtain the total removed.


In [ ]:
filter_reasons = cleaning_details["filter_reasons"]
display(format_cleaning_preview(filter_reasons))


### Sequential filtering flow

Each row counts newly removed orders and all their item rows. The ordered steps prevent double counting; the final row gives the retained totals.


In [ ]:
filter_flow = cleaning_details["filter_flow"]
with pd.option_context("display.max_columns", None):
    display(format_cleaning_preview(filter_flow))


### Excluded-order records

The complete `excluded_orders` table keeps every removed order ID, its item count and all applicable reason flags. The preview is limited to ten orders.


In [ ]:
excluded_orders = cleaning_details["excluded_orders"]
print(f"Excluded orders: {len(excluded_orders):,}; first ten shown")
with pd.option_context("display.max_columns", None):
    display(format_cleaning_preview(excluded_orders))


## 6. Retained dataset

The final table uses the 25 original columns. Quality flags support the inspection and exclusion tables; they are not included in the persisted clean table. Prices, freight, optional nulls and the original values of retained items are preserved, with city whitespace normalized.


### Before-and-after dimensions


In [ ]:
display(format_cleaning_preview(cleaning_details["dataset_summary"]))


### Final output validation

Verify the exact eligible order set, complete retained orders, original row order, schema, dtypes and permitted retained values against the full source. Confirm that the loaded interim input still matches its baseline.


In [ ]:
display(format_cleaning_preview(cleaning_details["output_checks"]))
assert_input_unchanged(df, input_baseline)


### Clean item preview

Readable labels and formats apply to this display copy. `cleaned_df` retains its canonical column names and calculation dtypes.


In [ ]:
cleaning_preview = format_cleaning_preview(cleaned_df, rows=10)
display(cleaning_preview)


## 7. Persistence and source preservation

Save `cleaned_order_items` in `processed` using the final schema. The saver prefers Parquet with CSV fallback. Read the actual successful format to verify this result even if another format already exists. The roundtrip comparison checks values, columns, dtypes and row order; stored files use item keys rather than pandas index labels.

For later independent reads, `load_data("cleaned_order_items", stage="processed")` prefers Parquet with CSV fallback. Add `file_format="csv"` to select CSV directly.


In [ ]:
saved_path = save_cleaned_data(cleaned_df)
reloaded_df = load_data(
    CLEANED_DATASET_NAME,
    stage="processed",
    file_format=saved_path.suffix.lstrip("."),
)
assert_cleaned_roundtrip(cleaned_df, reloaded_df)
print(f"Verified processed output: {saved_path}")


### Confirm the loaded source remains unchanged


In [ ]:
assert_input_unchanged(df, input_baseline)
print("Source preservation and processed roundtrip checks passed.")
